In [ ]:
# First we need to preprocess the data using Email Preprocessor Function
import numpy as np
import pandas as pd
import pickle
import re
from scipy.sparse import hstack, csr_matrix


class EmailPreprocessor:
    def __init__(self):
        self.free_email_domains = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'live.com']
        
    def process(self, raw_text):
        if not isinstance(raw_text, str): return {} 
        sender_match = re.search(r"From:\s*(.+)", raw_text)
        subject_match = re.search(r"Subject:\s*(.+)", raw_text)
        sender = sender_match.group(1).strip() if sender_match else ""
        subject = subject_match.group(1).strip() if subject_match else ""
        body = re.sub(r"From:.*?\n", "", raw_text, count=1)
        body = re.sub(r"Subject:.*?\n", "", body, count=1).strip()
        url_pattern = r"(https?://\S+|www\.\S+|\b\w+\.(?:com|net|org|info|biz|xyz|tk)\b)"
        urls = re.findall(url_pattern, raw_text.lower())
        full_text = f"{subject} {body}"
        clean_text = re.sub(url_pattern, " url ", full_text).lower()
        clean_text = re.sub(r'\s+', ' ', clean_text).strip()
        
        email_length = len(full_text) if len(full_text) > 0 else 1
        num_exclamation = full_text.count('!')
        upper_case_count = sum(1 for c in full_text if c.isupper())
        upper_case_ratio = upper_case_count / email_length
        num_urls = len(urls)
        url_to_text_ratio = num_urls / email_length
        
        
        has_generic_greeting = 1 if re.search(r"(dear|hi|hello)\s+(customer|user|winner|friend|student)", full_text.lower()) else 0
        domain = sender.split('@')[-1].lower() if '@' in sender else ""
        is_free_email = 1 if domain in self.free_email_domains else 0
        
        return {
            "Sender": sender, "Title": subject, "num_urls": num_urls,
            "email_length": email_length, "num_exclamation": num_exclamation,
            "upper_case_ratio": round(upper_case_ratio, 4), "url_to_text_ratio": round(url_to_text_ratio, 6),
            "has_generic_greeting": has_generic_greeting, "is_free_email": is_free_email,
            "full_clean_text": clean_text
        }

preprocessor = EmailPreprocessor()
print("Email Preprocessing completed")

Email Preprocessing completed


In [62]:


# Load the models and preprocessors
with open("../models/vectorizer.pkl", "rb") as f: vectorizer = pickle.load(f)
with open("../models/scaler.pkl", "rb") as f: scaler = pickle.load(f)
with open("../models/label_encoder.pkl", "rb") as f: label_encoder = pickle.load(f)

# Load the XGBoost and LightGBM models
with open("../models/xgb_model.pkl", "rb") as f: xgb_model = pickle.load(f)
with open("../models/lgb_model.pkl", "rb") as f: lgb_model = pickle.load(f)

print("Load the models successfully")


Load the models successfully


In [63]:
def Process_email(raw_email):
    feats = preprocessor.process(raw_email)

    # Convert text into number
    X_text = vectorizer.transform([feats['full_clean_text']])

    # Take out the numeric features
    numeric_features = ['num_urls', 'email_length', 'num_exclamation',
                        'upper_case_ratio', 'url_to_text_ratio',
                        'has_generic_greeting', 'is_free_email']
    X_num = [[feats[k] for k in numeric_features]]
    X_num_scaled = scaler.transform(X_num)

    # Build the final feature matrix
    X_final = hstack([X_text, csr_matrix(X_num_scaled)])

    # Ensemble the prediction
    prob_xgb = xgb_model.predict_proba(X_final.toarray())[0]
    prob_lgb = lgb_model.predict_proba(X_final.toarray())[0]
    final_prob = (prob_xgb + prob_lgb) / 2

    pred_idx = int(np.argmax(final_prob))
    label = label_encoder.inverse_transform([pred_idx])[0].upper()
    confidence = final_prob[pred_idx] * 100

    print("=" * 50)
    print(f"Label: {label}")
    print(f"Confidence: {confidence:.2f}%")
    print("-" * 30)
    print(f"Number of link:{feats['num_urls']}")
    print(f"Have email free?:  {'Yes' if feats['is_free_email'] else 'No'}")
    print(f"Exclamation mark:  {feats['num_exclamation']}")
    print("=" * 50)

In [ ]:
email = """From: giang.vien@swin.edu.au
Subject: Project Meeting Tomorrow

Hi Khang,
Please remember to prepare your presentation for the spam detection model tomorrow at 9 AM. 
Bring your laptop and the final report.
See you.
"""
Process_email(email)


In [ ]:
email = """From: admin-update@free-money.tk
Subject: URGENT: TUITION PAYMENT OVERDUE!!!

Dear Student 3416,
Your university account will be LOCKED in 24 hours!!! 
You MUST click here to update your billing information immediately to avoid suspension!!! 
http://fake-swinburne-login.xyz/update
http://another-scam-link.biz

DO IT NOW!!!
"""
Process_email(email)